# CLIP Model Comparison + Multi-Image/Video/Text Fusion

Two things this notebook answers:

1. **Is `laion/CLIP-ViT-H-14-laion2B-s32B-b79K` actually better than the production baseline
   (`openai/clip-vit-base-patch16`) for Hindu devotional content — and is it fast enough?**
   Production (`aikyam-vision`) runs CLIP on **CPU only** (see its Dockerfile — it explicitly
   installs the CPU-only torch wheel). ViT-H-14 has ~9x the parameters of ViT-B-16, so a GPU
   number here is not the number that matters for a go/no-go decision — this notebook measures
   both GPU and forced-CPU latency.

2. **How does one post with multiple images + a video + text become one embedding?**
   Part D builds that fusion and sanity-checks it.

Runtime: GPU (for convenience/speed on the accuracy pass); the CPU latency cell forces CPU
regardless of runtime, to mirror production.

In [ ]:
!pip install -q "transformers==4.46.3"

## 1. Load both models
Both load through the same generic `CLIPModel`/`CLIPProcessor` HF classes — this mirrors
`vision/clip_model.py` in aikyam-vision exactly, so results here transfer directly to a
config swap in production (`AIKYAM_CLIP_MODEL_NAME`).

In [ ]:
import torch
from transformers import CLIPModel, CLIPProcessor

GPU_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("GPU available:", GPU_DEVICE)

MODELS = {
    "baseline (openai/clip-vit-base-patch16)": "openai/clip-vit-base-patch16",
    "candidate (laion/CLIP-ViT-H-14-laion2B-s32B-b79K)": "laion/CLIP-ViT-H-14-laion2B-s32B-b79K",
}

loaded = {}
for label, name in MODELS.items():
    print(f"Loading {name} ...")
    proc = CLIPProcessor.from_pretrained(name, use_fast=False)
    model = CLIPModel.from_pretrained(name).to(GPU_DEVICE).eval()
    n_params = sum(p.numel() for p in model.parameters())
    loaded[label] = {"name": name, "proc": proc, "model": model, "params": n_params}
    print(f"  {n_params/1e6:.0f}M params")

## 2. Production cluster prompts (copied verbatim from `vision/clip_model.py`)
Scoring must use the exact same prompt set production uses — softmax is computed jointly across
*all* prompts, so a different/smaller prompt list would change the scores and make this
comparison meaningless.

In [ ]:
# Paste the full CLUSTER_PROMPTS dict from aikyam-vision/vision/clip_model.py here.
# Keeping it in one place (not re-typed) avoids the two copies drifting apart --
# copy/paste the current version before each real test run.
CLUSTER_PROMPTS = {
    "shiva_iconography": [
        "a photograph of a sacred Shivalinga idol decorated with bilva leaves and water offerings inside a temple",
        "a photograph of Lord Shiva meditating in a serene posture with a Trishul trident and crescent moon",
        "a sacred stone Shivalinga with milk abhishekam, flowers, and holy ash tripundra marking",
        "a small Shivalinga idol placed in a temple background",
        "a photo of Lord Shiva holding a trishul trident"
    ],
    "krishna_iconography": [
        "a photograph of the blue-skinned Hindu deity Lord Krishna playing a bamboo flute with a peacock feather",
        "a photograph of Radha and Krishna, featuring a dark-skinned male playing a flute with a female companion",
        "a photograph of a child Bal Krishna deity figure with blue skin eating white butter from a clay pot",
        "a photograph of the three traditional wooden Jagannath idols with large round eyes and painted faces",
        "a small idol of Lord Krishna placed in a temple background"
    ],
    "vishnu_iconography": [
        "a photograph of the four-armed Hindu deity Lord Vishnu holding a conch shell, sudarshana chakra, and mace",
        "a photograph of blue-skinned Lord Vishnu reclining on the giant multi-headed coiled serpent Sheshnag",
        "a photograph of a Lord Vishnu statue wearing a gold crown and fresh flower garlands",
        "a small idol of Lord Vishnu placed in a temple background",
        "a photo of Lord Vishnu inside a temple sanctum"
    ],
    "venkateswara_iconography": [
        "a photograph of a standing Tirupati Balaji idol with a large white V-shaped forehead tilak and tall gold crown",
        "a photograph of a Lord Venkateswara idol wearing a tall gold mukut crown and thick marigold flower garlands",
        "a photograph of a standing dark stone Tirupati Balaji idol inside a temple sanctum",
        "a small idol of Tirupati Balaji placed in a temple background",
        "a photo of Lord Venkateswara with white V tilak and garlands"
    ],
    "rama_iconography": [
        "a photograph of Lord Rama holding a large strung bow and arrow",
        "a photograph of a Ram Darbar idol set featuring Lord Rama, Goddess Sita, Lakshmana, and a kneeling Hanuman",
        "a photograph of the Ayodhya Ram Lalla idol adorned in gold holding a bow",
        "a small idol of Lord Rama placed in a temple background",
        "a photo of Shri Ram and Sita Devi adorned in royal traditional attire"
    ],
    "hanuman_iconography": [
        "a photograph of Lord Hanuman holding a golden gada mace in a devotional posture",
        "a photograph of a saffron-colored Lord Hanuman idol covered in bright orange sindoor carrying Sanjeevani mountain",
        "a temple idol of Bajrangbali Hanuman folded hands in devotion to Ram",
        "a small idol of Lord Hanuman placed in a temple background",
        "a photo of muscular monkey-faced Lord Hanuman wearing a gold crown"
    ],
    "ganesha_iconography": [
        "a photograph of a Lord Ganesha idol with elephant head, holding modak sweets and lotus flower",
        "a photograph of the elephant-headed Hindu deity Lord Ganesha with four arms sitting on a throne",
        "a photograph of a Ganapati idol wearing an ornate gold mukut crown and vibrant marigold flower garlands",
        "a small idol of Lord Ganesha placed in a temple background",
        "a photo of a seated elephant-headed deity Ganesha accompanied by mooshak mouse"
    ],
    "devi_iconography": [
        "a photograph of Goddess Durga riding a lion holding weapons in multiple arms",
        "a photograph of Goddess Lakshmi sitting on a pink lotus flower holding gold coins",
        "a photograph of Goddess Saraswati in a pure white sari playing a long stringed veena instrument",
        "a small idol of Goddess Durga, Lakshmi, or Saraswati in a temple background",
        "a photograph of a female Hindu goddess idol wearing an ornate gold crown and sari"
    ],
    "puja_rituals_and_temples": [
        "a close-up photograph of hands holding a brass ritual lamp with multiple lit burning camphor flames during aarti",
        "a photograph of a person pouring white milk from a brass pot over a stone deity idol during abhishekam ritual",
        "a photograph of an ancient stone Hindu temple gopuram tower with detailed traditional carvings",
        "a photograph of the sanctum sanctorum garbha griha of a Hindu temple illuminated by oil lamps",
        "a wide photograph of a large crowd of devotees standing in a river performing Chhath puja prayers with folded hands",
        "a photograph of devotees offering fruit and diya baskets on a riverside ghat during a religious ritual",
        "a photograph of covered metal processional utsava idols being carried in a temple festival procession",
        "a wide photograph of pilgrims queueing in a long line at a temple entrance gopuram for darshan",
        "a photograph of a Hindu temple exterior courtyard with devotees walking past ornately carved gateway towers",
        "a photograph of a priest performing a ritual with multiple devotees gathered around a temple altar"
    ],
    "festivals_and_celebrations": [
        "a photograph of lit brass diya oil lamps arranged on floor during Diwali festival",
        "a photograph of people celebrating Holi festival with bright organic gulal color powders",
        "a photograph of a colorful rangoli floor art design made with powders and flower petals"
    ],
    "safe_everyday_life": [
        "a normal everyday photograph of ordinary daily life with no religious or unsafe content",
        "a photograph of people walking on a city street, shops and vehicles in the background",
        "a portrait photograph of a person or small group in casual everyday clothing",
        "a photograph of a scenic mountain, river, or natural landscape with no people or buildings",
        "a photograph of a building or cityscape at night lit up with ordinary decorative or architectural lighting",
        "a photograph of a plate of vegetarian food such as pizza, pasta, salad, fries, or vegetable dishes"
    ],
    "modern_vehicles_and_tech": [
        "a photograph of a car, motorcycle, or other vehicle on a street or road",
        "a photograph of a modern smartphone, laptop, or other electronic device",
        "a photograph of a race car or sports car with visible sponsor logos and text",
        "a photograph of a bus, tram, or train with passengers boarding or riding",
        "a photograph of an old, rusted, or abandoned vehicle outdoors"
    ],
    "nudity_and_sexual_content": [
        "A fully nude person with no clothing covering the body.",
        "Explicit sexual activity or intimate physical contact between nude people.",
        "A close-up photograph focused on exposed genitals or explicit intimate body parts.",
        "Pornographic or sexually explicit photographic imagery.",
        "A person deliberately exposing intimate body parts in a sexually suggestive manner."
    ],
    "weapons_and_firearms": [
        "a close-up photograph of a handgun or pistol resting on a surface or held in a hand",
        "a photograph of a rifle or shotgun visible in the frame, held or resting against a wall",
        "a photograph of a large knife, machete, or sword blade held or displayed",
        "a photograph of a person holding or carrying a firearm",
        "a photograph of a collection of weapons such as guns and knives displayed together"
    ],
}
# NOTE: trimmed to the clusters most relevant to this comparison (devotional + a few
# safety/neutral controls) to keep this notebook readable. Swap in the FULL dict from
# clip_model.py before treating any result here as a real production decision --
# a smaller prompt set changes the softmax denominator and shifts every score.
print(f"{len(CLUSTER_PROMPTS)} clusters, {sum(len(v) for v in CLUSTER_PROMPTS.values())} prompts")

In [ ]:
def _as_tensor(feats):
    """Some transformers versions wrap get_text/image_features output in a
    ModelOutput instead of returning the tensor directly -- unwrap defensively
    so this works regardless of the installed version."""
    if torch.is_tensor(feats):
        return feats
    if hasattr(feats, "pooler_output") and feats.pooler_output is not None:
        return feats.pooler_output
    if hasattr(feats, "last_hidden_state"):
        return feats.last_hidden_state[:, 0, :]
    return feats[0]


def get_text_features(entry, device):
    """Cache normalized text features for every prompt, once per model."""
    if "text_features" in entry and entry["text_device"] == device:
        return entry["cluster_ids"], entry["text_features"]

    cluster_ids, prompts = [], []
    for cluster_id, cluster_prompts in CLUSTER_PROMPTS.items():
        for p in cluster_prompts:
            cluster_ids.append(cluster_id)
            prompts.append(p)

    inputs = entry["proc"](text=prompts, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        feats = _as_tensor(entry["model"].get_text_features(**inputs))
        feats = feats / feats.norm(dim=-1, keepdim=True)

    entry["cluster_ids"], entry["text_features"], entry["text_device"] = cluster_ids, feats, device
    return cluster_ids, feats


def cluster_scores_from_image(entry, image, device):
    """Mirrors _cluster_scores_from_prompt_probs in vision/clip_model.py."""
    cluster_ids, text_features = get_text_features(entry, device)
    inputs = entry["proc"](images=image, return_tensors="pt").to(device)
    with torch.no_grad():
        image_features = _as_tensor(entry["model"].get_image_features(**inputs))
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)
        logit_scale = entry["model"].logit_scale.exp()
        logits = logit_scale * (image_features @ text_features.T)
    probs = logits[0].softmax(dim=0).tolist()

    totals, counts = {}, {}
    for cid, p in zip(cluster_ids, probs):
        totals[cid] = totals.get(cid, 0.0) + p
        counts[cid] = counts.get(cid, 0) + 1
    scores = {cid: totals[cid] / counts[cid] for cid in totals}
    return dict(sorted(scores.items(), key=lambda kv: -kv[1])), image_features[0].cpu().tolist()

## 3. Part A — accuracy on real data (Kaggle Indian Temples dataset)
Reuses the same dataset as `Temple_Linear_Probe.ipynb`. Every image in this dataset is Hindu
temple architecture, so a good model should rank `puja_rituals_and_temples` highest and
*not* confuse it with `safe_everyday_life` / `modern_vehicles_and_tech`.

In [ ]:
!pip install -q kaggle
from google.colab import files
files.upload()  # kaggle.json
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d tarundalal/indian-temples-dataset -p data --unzip

In [ ]:
import os, random
from PIL import Image

def find_dataset_root(base="data"):
    for dirpath, dirnames, _ in os.walk(base):
        if len(dirnames) > 1:
            return dirpath
    return base

root = find_dataset_root()
all_images = []
for dirpath, _, filenames in os.walk(root):
    for f in filenames:
        if f.lower().endswith((".jpg", ".jpeg", ".png")):
            all_images.append(os.path.join(dirpath, f))

random.seed(42)
SAMPLE_N = 150  # keep the accuracy pass fast; raise for a more rigorous read
sample_paths = random.sample(all_images, min(SAMPLE_N, len(all_images)))
print(f"{len(all_images)} images found, evaluating on {len(sample_paths)}")

In [ ]:
import time

TARGET_CLUSTER = "puja_rituals_and_temples"
results = {label: {"correct_top1": 0, "target_score_sum": 0.0, "n": 0} for label in loaded}

for path in sample_paths:
    try:
        image = Image.open(path).convert("RGB")
    except Exception:
        continue
    for label, entry in loaded.items():
        scores, _ = cluster_scores_from_image(entry, image, GPU_DEVICE)
        top_cluster = next(iter(scores))
        results[label]["correct_top1"] += int(top_cluster == TARGET_CLUSTER)
        results[label]["target_score_sum"] += scores.get(TARGET_CLUSTER, 0.0)
        results[label]["n"] += 1

print(f"{'model':55s} {'top1_acc':>10s} {'mean_target_score':>20s}")
for label, r in results.items():
    acc = r["correct_top1"] / r["n"]
    mean_score = r["target_score_sum"] / r["n"]
    print(f"{label:55s} {acc:10.3f} {mean_score:20.4f}")

## 4. Part B — deity/yoga probe images (edit this cell)
`Part A` only proves the model recognizes *temple architecture*. Your actual concern is
individual deities and yoga. Paste in a handful of image URLs or local paths you know the
correct cluster for (use your own post images / a small curated set — don't trust
guessed-at web URLs here, dead links just get skipped silently).

In [ ]:
import requests
from io import BytesIO

# EDIT ME: (image_url_or_local_path, expected_cluster_id)
PROBE_IMAGES = [
    # ("https://example.com/shiva1.jpg", "shiva_iconography"),
    # ("/content/my_yoga_photo.jpg", "safe_everyday_life"),  # no yoga cluster exists yet -- see notes
]

def load_image(src):
    if src.startswith("http"):
        resp = requests.get(src, timeout=10, headers={"User-Agent": "Mozilla/5.0"})
        return Image.open(BytesIO(resp.content)).convert("RGB")
    return Image.open(src).convert("RGB")

if not PROBE_IMAGES:
    print("No probe images set -- add some to PROBE_IMAGES above and re-run this cell.")
else:
    for src, expected in PROBE_IMAGES:
        try:
            image = load_image(src)
        except Exception as e:
            print(f"skip {src}: {e}")
            continue
        print(f"\n{src}  (expected: {expected})")
        for label, entry in loaded.items():
            scores, _ = cluster_scores_from_image(entry, image, GPU_DEVICE)
            top3 = list(scores.items())[:3]
            print(f"  {label}: {top3}")

## 5. Part C — latency (GPU, then forced CPU to mirror production)
Production runs CLIP on CPU only. If the candidate model is meaningfully more accurate but
too slow on CPU for the Kafka worker's throughput needs, that's a real go/no-go input, not
just a nice-to-know.

In [ ]:
def benchmark(entry, device, n=20):
    entry["model"].to(device)
    if "text_features" in entry:
        del entry["text_features"]  # force recompute on this device
    image = Image.open(sample_paths[0]).convert("RGB")
    cluster_scores_from_image(entry, image, device)  # warmup

    start = time.perf_counter()
    for path in sample_paths[:n]:
        img = Image.open(path).convert("RGB")
        cluster_scores_from_image(entry, img, device)
    elapsed = time.perf_counter() - start
    return (elapsed / n) * 1000  # ms/image

print(f"{'model':55s} {'GPU ms/img':>12s} {'CPU ms/img':>12s}")
for label, entry in loaded.items():
    gpu_ms = benchmark(entry, GPU_DEVICE) if GPU_DEVICE == "cuda" else float("nan")
    cpu_ms = benchmark(entry, "cpu")
    entry["model"].to(GPU_DEVICE)  # restore for later cells
    if "text_features" in entry:
        del entry["text_features"]
    print(f"{label:55s} {gpu_ms:12.1f} {cpu_ms:12.1f}")

## 6. Part D — one embedding for 3 images + 1 video + text
CLIP has no multi-image or video input -- each image/frame/text string gets its own vector.
This builds one consolidated post embedding via mean-pooling + L2-normalize, then sanity-checks
it against a few text queries: a fused vector should score highest against the query that best
matches the post's actual content.

In [ ]:
import numpy as np
try:
    import cv2
except ImportError:
    !pip install -q opencv-python-headless
    import cv2

def encode_image(entry, image, device):
    inputs = entry["proc"](images=image, return_tensors="pt").to(device)
    with torch.no_grad():
        feat = _as_tensor(entry["model"].get_image_features(**inputs))
        feat = feat / feat.norm(dim=-1, keepdim=True)
    return feat[0].cpu().numpy()

def encode_text(entry, text, device):
    inputs = entry["proc"](text=[text], return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        feat = _as_tensor(entry["model"].get_text_features(**inputs))
        feat = feat / feat.norm(dim=-1, keepdim=True)
    return feat[0].cpu().numpy()

def sample_video_frames(video_path, n_frames=5):
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release()
        return []
    idxs = np.linspace(0, total - 1, min(n_frames, total), dtype=int)
    frames = []
    for idx in idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ok, frame = cap.read()
        if ok:
            frames.append(Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))
    cap.release()
    return frames

def build_post_embedding(entry, device, image_paths=(), video_path=None, caption="", n_video_frames=5):
    """Mean-pool + L2-normalize every image/frame/text vector into one post embedding."""
    vectors = []
    for p in image_paths:
        vectors.append(encode_image(entry, Image.open(p).convert("RGB"), device))
    if video_path:
        for frame in sample_video_frames(video_path, n_video_frames):
            vectors.append(encode_image(entry, frame, device))
    if caption:
        vectors.append(encode_text(entry, caption, device))

    if not vectors:
        return None
    pooled = np.mean(vectors, axis=0)
    pooled = pooled / np.linalg.norm(pooled)
    return pooled

In [ ]:
# EDIT ME: point at a real post -- 3 images + (optionally) 1 video + a caption.
POST_IMAGES = sample_paths[:3]         # placeholder: 3 temple images from the Kaggle set
POST_VIDEO = None                      # e.g. "/content/my_video.mp4" -- upload via files.upload()
POST_CAPTION = "Om Namah Shivaya, darshan at the temple today"

candidate = loaded["candidate (laion/CLIP-ViT-H-14-laion2B-s32B-b79K)"]
post_vec = build_post_embedding(
    candidate, GPU_DEVICE,
    image_paths=POST_IMAGES, video_path=POST_VIDEO, caption=POST_CAPTION,
)
print("Consolidated post embedding shape:", post_vec.shape)

queries = [
    "a photograph of a Hindu temple",
    "a photograph of Lord Shiva",
    "a person doing a yoga pose",
    "a plate of food",
    "a car on a highway",
]
print("\nCosine similarity of the fused post embedding to each query:")
for q in queries:
    q_vec = encode_text(candidate, q, GPU_DEVICE)
    sim = float(np.dot(post_vec, q_vec))
    print(f"  {sim:.4f}  {q}")

Expect the temple/Shiva queries to score highest for a devotional post -- that confirms
mean-pooling preserved the post's meaning well enough for retrieval. If the fused vector
scores roughly the same against everything (no clear winner), pooling washed out the signal
and late-fusion (keep vectors separate, max-score at query time) is worth the extra
complexity instead.

**Portability note**: this embedding is only comparable to other vectors built the same way
-- same model (`laion/CLIP-ViT-H-14...`), same fusion recipe (mean-pool + L2-normalize). Tag
every stored vector with which model/recipe produced it; never compare vectors across a model
swap without re-embedding.

## 7. Decision summary
Fill in after running Parts A-C:

| | baseline (ViT-B-16) | candidate (ViT-H-14) |
|---|---|---|
| Temple top-1 accuracy | | |
| Mean target-cluster score | | |
| CPU ms/image | | |
| Params | ~150M | ~2.5B |

Rule of thumb: if candidate CPU latency is >3-4x baseline, it likely breaks the Kafka worker's
throughput at production post volume -- consider `laion/CLIP-ViT-B-16-laion2B` instead (same
size class as today, LAION-trained data, much smaller accuracy-per-ms tradeoff) before
committing to ViT-H-14 in the worker.